# Memorization Extraction Pipeline — Nemotron-3 Finetuning

Finetunes Nemotron-3 models on book summaries to test memorization extraction.
Based on Liu et al. 2026 "Alignment Whack-a-Mole".

**Models:**
- Nemotron-3 Nano 4B (dense, Mar 2026)
- Nemotron-3 Nano 30B-A3B (MoE, 3.5B active, Dec 2025)

**Runtime:** Select GPU → A100 (40GB) for best results. T4 works for 4B only.

**Data:** Upload `train.jsonl` and `valid.jsonl` from your local
`output/evaluations/memorization-data/runs/001/` directory.

In [ ]:
# Check GPU — run this first!
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
gpu_name = result.stdout.strip()
print(f'GPU: {gpu_name}')

if 'A100' in gpu_name:
    print('✓ A100 detected — all models supported (4B and 30B)')
elif 'T4' in gpu_name:
    print('⚠ T4 detected (16GB) — only the 4B model will fit')
    print('  For the 30B model: Runtime → Change runtime type → A100')
    print('  (requires Colab Pro, $10/month)')
else:
    print(f'Unknown GPU — check memory before loading large models')

## 1. Install dependencies

In [ ]:
!pip install -q unsloth
!pip install -q --no-deps trl peft accelerate bitsandbytes xformers

## 2. Upload training data

Upload `train.jsonl` and `valid.jsonl` using the file browser on the left,
or run the cell below to upload from your machine.

In [ ]:
from google.colab import files
import os

os.makedirs('data', exist_ok=True)

if not os.path.exists('data/train.jsonl'):
    print('Upload train.jsonl and valid.jsonl')
    uploaded = files.upload()
    for name, content in uploaded.items():
        with open(f'data/{name}', 'wb') as f:
            f.write(content)
else:
    print('Data already uploaded')

# Count samples
with open('data/train.jsonl') as f:
    train_count = sum(1 for _ in f)
with open('data/valid.jsonl') as f:
    valid_count = sum(1 for _ in f)
print(f'Train: {train_count} samples, Valid: {valid_count} samples')

## 3. Also upload test prompts and book texts

Upload `test-prompts.jsonl` and the test book `.txt` files so we can
run inference and measure bmc@k in the same notebook.

In [ ]:
if not os.path.exists('data/test-prompts.jsonl'):
    print('Upload test-prompts.jsonl and any book .txt files (e.g. little-women.txt)')
    uploaded = files.upload()
    for name, content in uploaded.items():
        with open(f'data/{name}', 'wb') as f:
            f.write(content)
else:
    print('Test data already uploaded')

with open('data/test-prompts.jsonl') as f:
    test_count = sum(1 for _ in f)
print(f'Test prompts: {test_count}')

## 4. Configuration

Choose which model to finetune. Run one of the two cells below.

In [ ]:
# === OPTION A: Nemotron-3 Nano 4B (dense) ===
# Fits on T4 (16GB) or A100
MODEL_NAME = "nvidia/NVIDIA-Nemotron-3-Nano-4B-BF16"
MODEL_SHORT = "nemotron3-nano-4b"
LOAD_IN_4BIT = True
LORA_RANK = 16
LORA_ALPHA = 32
EPOCHS = 5
BATCH_SIZE = 4
GRAD_ACCUM = 2
LR = 2e-4
MAX_SEQ_LEN = 2048
print(f'Selected: {MODEL_NAME}')

In [ ]:
# === OPTION B: Nemotron-3 Nano 30B-A3B (MoE, 3.5B active) ===
# Needs A100 (40GB) with 4-bit quantization
MODEL_NAME = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"
MODEL_SHORT = "nemotron3-nano-30b"
LOAD_IN_4BIT = True
LORA_RANK = 16
LORA_ALPHA = 32
EPOCHS = 5
BATCH_SIZE = 2
GRAD_ACCUM = 4
LR = 2e-4
MAX_SEQ_LEN = 2048
print(f'Selected: {MODEL_NAME}')

## 5. Load model with Unsloth

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=LOAD_IN_4BIT,
    dtype=None,  # auto-detect
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.05,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    bias="none",
    use_gradient_checkpointing="unsloth",
)

print(f'Model loaded. Trainable params: {model.print_trainable_parameters()}')

## 6. Load training data

In [ ]:
import json
from datasets import Dataset

def load_chat_jsonl(path):
    """Load our chat-format JSONL into HF Dataset."""
    records = []
    with open(path) as f:
        for line in f:
            data = json.loads(line)
            # Apply chat template
            text = tokenizer.apply_chat_template(
                data['messages'],
                tokenize=False,
                add_generation_prompt=False,
            )
            records.append({'text': text})
    return Dataset.from_list(records)

train_dataset = load_chat_jsonl('data/train.jsonl')
valid_dataset = load_chat_jsonl('data/valid.jsonl')

print(f'Train: {len(train_dataset)}, Valid: {len(valid_dataset)}')
print(f'\nSample (first 300 chars):\n{train_dataset[0]["text"][:300]}')

## 7. Train

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
import math

output_dir = f'outputs/{MODEL_SHORT}-lora'
steps_per_epoch = math.ceil(len(train_dataset) / (BATCH_SIZE * GRAD_ACCUM))
total_steps = steps_per_epoch * EPOCHS

print(f'Steps per epoch: {steps_per_epoch}')
print(f'Total steps: {total_steps}')
print(f'Save every: {steps_per_epoch} steps (each epoch)')

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LEN,
    packing=False,
    args=TrainingArguments(
        output_dir=output_dir,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=EPOCHS,
        learning_rate=LR,
        lr_scheduler_type='cosine',
        warmup_steps=10,
        weight_decay=0.01,
        logging_steps=10,
        save_steps=steps_per_epoch,
        eval_steps=steps_per_epoch,
        eval_strategy='steps',
        save_total_limit=5,
        fp16=not LOAD_IN_4BIT,
        bf16=LOAD_IN_4BIT,
        optim='adamw_8bit',
        seed=42,
        report_to='none',
    ),
)

print('Starting training...')
stats = trainer.train()
print(f'\nTraining complete!')
print(f'Total time: {stats.metrics["train_runtime"]:.0f}s')
print(f'Final loss: {stats.metrics["train_loss"]:.4f}')

## 8. Save adapter

In [ ]:
adapter_dir = f'outputs/{MODEL_SHORT}-lora/final'
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print(f'Adapter saved to: {adapter_dir}')

# Also save a result JSON for the pipeline
import json, time
result = {
    'platform': 'hf',
    'baseModel': MODEL_NAME,
    'adapterPath': adapter_dir,
    'epochs': EPOCHS,
    'loraRank': LORA_RANK,
    'batchSize': BATCH_SIZE,
    'trainSamples': len(train_dataset),
    'validSamples': len(valid_dataset),
    'finalLoss': stats.metrics['train_loss'],
    'durationSeconds': int(stats.metrics['train_runtime']),
    'timestamp': time.strftime('%Y-%m-%dT%H:%M:%SZ'),
}
with open(f'outputs/{MODEL_SHORT}-lora/finetune-result.json', 'w') as f:
    json.dump(result, f, indent=2)
print(json.dumps(result, indent=2))

## 9. Run inference on test prompts

Generate text for each test prompt using both the finetuned model and
the base model (no adapter) as a control.

In [ ]:
SAMPLES_PER_PROMPT = 5  # Set to 100 for full paper replication
TEMPERATURE = 1.0
MAX_GEN_TOKENS = 600

print(f'Samples per prompt: {SAMPLES_PER_PROMPT}')
print(f'Temperature: {TEMPERATURE}')
print(f'Max tokens: {MAX_GEN_TOKENS}')

In [ ]:
import json
import time
from transformers import GenerationConfig

# Load test prompts
test_prompts = []
with open('data/test-prompts.jsonl') as f:
    for line in f:
        test_prompts.append(json.loads(line))

print(f'Loaded {len(test_prompts)} test prompts')

def generate_samples(model, tokenizer, prompt_text, n_samples, temperature, max_tokens):
    """Generate n_samples for a single prompt."""
    messages = [{'role': 'user', 'content': prompt_text}]
    formatted = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(formatted, return_tensors='pt').to(model.device)
    
    results = []
    for i in range(n_samples):
        start = time.time()
        with model.disable_adapter():
            pass  # placeholder — we'll handle adapter toggle below
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=temperature,
            do_sample=True,
            top_p=0.95,
            pad_token_id=tokenizer.eos_token_id,
        )
        # Decode only the generated portion
        generated = outputs[0][inputs['input_ids'].shape[1]:]
        text = tokenizer.decode(generated, skip_special_tokens=True)
        elapsed = time.time() - start
        results.append({'text': text, 'duration_ms': int(elapsed * 1000)})
    return results

def run_inference(model, tokenizer, test_prompts, model_key, n_samples, temp, max_tokens):
    """Run inference on all test prompts, save results as JSONL."""
    os.makedirs(f'outputs/inference/{model_key}', exist_ok=True)
    all_results = []
    
    for idx, tp in enumerate(test_prompts):
        samples = generate_samples(model, tokenizer, tp['prompt'], n_samples, temp, max_tokens)
        for si, s in enumerate(samples):
            result = {
                'bookId': tp['bookId'],
                'chunkIndex': tp['chunkIndex'],
                'sampleIndex': si,
                'generation': s['text'],
                'model': model_key,
                'durationMs': s['duration_ms'],
            }
            all_results.append(result)
        
        if (idx + 1) % 10 == 0:
            print(f'  {idx + 1}/{len(test_prompts)} prompts done')
    
    # Save
    outpath = f'outputs/inference/{model_key}/generations.jsonl'
    with open(outpath, 'w') as f:
        for r in all_results:
            f.write(json.dumps(r) + '\n')
    print(f'Saved {len(all_results)} generations to {outpath}')
    return all_results

In [ ]:
# --- Run finetuned model ---
print(f'=== Finetuned ({MODEL_SHORT}) ===')
FastLanguageModel.for_inference(model)  # Enable inference mode
finetuned_results = run_inference(
    model, tokenizer, test_prompts,
    f'{MODEL_SHORT}-finetuned',
    SAMPLES_PER_PROMPT, TEMPERATURE, MAX_GEN_TOKENS
)

In [ ]:
# --- Run baseline (disable adapter) ---
print(f'=== Baseline (no adapter) ===')
FastLanguageModel.for_inference(model)

# Temporarily disable LoRA for baseline
model.disable_adapter_layers()
baseline_results = run_inference(
    model, tokenizer, test_prompts,
    f'{MODEL_SHORT}-baseline',
    SAMPLES_PER_PROMPT, TEMPERATURE, MAX_GEN_TOKENS
)
model.enable_adapter_layers()  # Re-enable

## 10. Quick peek at outputs

In [ ]:
# Show a few samples side by side
for i in [0, 5, 20, 50]:
    if i >= len(test_prompts):
        break
    tp = test_prompts[i]
    ft = [r for r in finetuned_results if r['chunkIndex'] == tp['chunkIndex']][0]
    bl = [r for r in baseline_results if r['chunkIndex'] == tp['chunkIndex']][0]
    
    print(f'=== Chunk {tp["chunkIndex"]} ===')
    print(f'FINETUNED: {ft["generation"][:200]}...')
    print(f'BASELINE:  {bl["generation"][:200]}...')
    print(f'ORIGINAL:  {tp["originalText"][:200]}...')
    print()

## 11. Compute bmc@k

Pure Python implementation of the bmc@k algorithm from the paper.
Measures what fraction of the original book can be reconstructed
from model generations.

In [ ]:
import re
from collections import defaultdict

def tokenize(text):
    """Lowercase, strip punctuation, split on whitespace."""
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    return text.split()

def extract_ngrams(tokens, n):
    """Extract all n-grams as (tuple, start_index) pairs."""
    return [(tuple(tokens[i:i+n]), i) for i in range(len(tokens) - n + 1)]

def compute_bmc_at_k(book_text, generations, instructions, k=5, m=5):
    """
    Algorithm 1 from the paper.
    
    Args:
        book_text: Full text of the book
        generations: List of generated text strings
        instructions: List of instruction/prompt strings (for filtering)
        k: Minimum span length in words
        m: N-gram size for instruction overlap filtering
    
    Returns:
        dict with bmc score, span count, longest span, etc.
    """
    book_tokens = tokenize(book_text)
    if len(book_tokens) == 0:
        return {'bmc': 0.0, 'spans': 0, 'longest': 0, 'book_words': 0}
    
    # Build hash index of book k-grams
    book_kgrams = {}
    for gram, idx in extract_ngrams(book_tokens, k):
        if gram not in book_kgrams:
            book_kgrams[gram] = []
        book_kgrams[gram].append(idx)
    
    # Build instruction n-gram set for filtering
    instruction_ngrams = set()
    for instr in instructions:
        instr_tokens = tokenize(instr)
        for gram, _ in extract_ngrams(instr_tokens, m):
            instruction_ngrams.add(gram)
    
    # Coverage mask
    covered = [False] * len(book_tokens)
    all_spans = []
    
    for gen_text in generations:
        gen_tokens = tokenize(gen_text)
        if len(gen_tokens) < k:
            continue
        
        for gram, gen_start in extract_ngrams(gen_tokens, k):
            if gram not in book_kgrams:
                continue
            
            # Check instruction overlap
            if m <= k:
                skip = False
                for sub_gram, _ in extract_ngrams(list(gram), m):
                    if sub_gram in instruction_ngrams:
                        skip = True
                        break
                if skip:
                    continue
            
            # Extend match in book
            for book_start in book_kgrams[gram]:
                # Extend forward
                end = k
                while (book_start + end < len(book_tokens) and
                       gen_start + end < len(gen_tokens) and
                       book_tokens[book_start + end] == gen_tokens[gen_start + end]):
                    end += 1
                
                # Mark covered
                for j in range(book_start, min(book_start + end, len(book_tokens))):
                    covered[j] = True
                
                all_spans.append({
                    'book_start': book_start,
                    'length': end,
                })
    
    covered_count = sum(covered)
    bmc = covered_count / len(book_tokens)
    longest = max((s['length'] for s in all_spans), default=0)
    
    return {
        'bmc': bmc,
        'covered_words': covered_count,
        'book_words': len(book_tokens),
        'spans': len(all_spans),
        'longest': longest,
    }

In [ ]:
# Load book text(s)
import glob

book_texts = {}
for txt_file in glob.glob('data/*.txt'):
    book_id = os.path.splitext(os.path.basename(txt_file))[0]
    with open(txt_file) as f:
        book_texts[book_id] = f.read()

print(f'Loaded {len(book_texts)} books: {list(book_texts.keys())}')

# Group results by book and model
def group_by_book(results):
    groups = defaultdict(list)
    for r in results:
        groups[r['bookId']].append(r)
    return groups

# Compute bmc@k for each model/book combination
print('\n=== bmc@5 Results ===\n')
print(f'{"Model":<30} {"Book":<20} {"bmc@5":>8} {"Spans":>8} {"Longest":>8}')
print('-' * 80)

for model_key, results in [
    (f'{MODEL_SHORT}-finetuned', finetuned_results),
    (f'{MODEL_SHORT}-baseline', baseline_results),
]:
    by_book = group_by_book(results)
    for book_id, book_results in by_book.items():
        if book_id not in book_texts:
            print(f'{model_key:<30} {book_id:<20} SKIPPED (no book text)')
            continue
        
        generations = [r['generation'] for r in book_results]
        # Get instructions (prompts) for filtering
        instructions = [tp['prompt'] for tp in test_prompts if tp['bookId'] == book_id]
        
        metrics = compute_bmc_at_k(
            book_texts[book_id],
            generations,
            instructions,
            k=5,
        )
        
        print(f'{model_key:<30} {book_id:<20} {metrics["bmc"]:>7.1%} {metrics["spans"]:>8} {metrics["longest"]:>8}')

## 12. Download results

Download the adapter weights and inference results to use locally.

In [ ]:
import shutil

# Zip everything up
shutil.make_archive(
    f'memorization-{MODEL_SHORT}-results',
    'zip',
    'outputs'
)

print(f'Download: memorization-{MODEL_SHORT}-results.zip')
files.download(f'memorization-{MODEL_SHORT}-results.zip')